In [1]:
import pandas as pd
import numpy as np
from pprint import pprint
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import RDF, SKOS, DC

### Define vocabulary graph and declare concept scheme

In [ ]:
voc = Graph()
concept_scheme = URIRef("https://vocabs.dariah.eu/resource-type/")
voc.add((concept_scheme, RDF.type, SKOS.ConceptScheme))
voc.add((concept_scheme, SKOS.prefLabel, Literal("SSHOMP Resource Type", lang='en')))
voc.add((concept_scheme, DC.title, Literal("SSHOMP Resource Type", lang='en')))
voc.add((concept_scheme, DC.creator, Literal("Laure Barbot")))
voc.add((concept_scheme, DC.creator, Literal("Massimiliano Carloni")))
voc.add((concept_scheme, DC.creator, Literal("Matej Ďurčo")))
voc.add((concept_scheme, DC.creator, Literal("Vera Maria Charvát")))
voc.add((concept_scheme, DC.creator, Literal("Michael Kurzmeier")))

<Graph identifier=N7f51e4dd947a42afa1da9772a07f4a33 (<class 'rdflib.graph.Graph'>)>

### Load vocabulary data from CSV

In [3]:
df = pd.read_csv('data.csv', skiprows=5, names=["top", "label_orig", "label_clean", "exact_match", "definition", "close_match", "comments", "uri"])

Replace all empty strings (or strings made just of spaces) with NaN, just to be sure

In [4]:
df = df.replace(r'^\s*$', np.nan, regex=True)

In [5]:
df.head()

,top,label_orig,label_clean,exact_match,definition,close_match,comments,uri
0,Semantic Resource,NaN,semantic resource,NaN,"""machine-actionable formalisation (represented...",NaN,NaN,NaN
1,NaN,Metadata schema,metadata schema,NaN,"""Resources for defining and organising metadat...",NaN,NaN,NaN
2,NaN,Ontology,ontology,NaN,"""Resources for formalising knowledge within a ...",NaN,NaN,NaN
3,NaN,Vocabulary,vocabulary,NaN,"""Set of terms or concepts used within a specif...",NaN,NaN,NaN
4,NaN,Thesaurus,thesaurus,NaN,"""Structured, controlled vocabularies where ter...",NaN,NaN,NaN


### Check topConcepts against sub-concepts

In [6]:
top_concepts = df.loc[df["top"].notna(), "label_clean"].tolist()
print(top_concepts)

['semantic resource', 'service', 'software', 'dataset', 'workflow', 'training material', 'publication']


In [7]:
sub_concepts = df.loc[df["label_orig"].notna(), "label_clean"].tolist()
print(sub_concepts)

['metadata schema', 'ontology', 'vocabulary', 'thesaurus', 'gazetteer', 'model', 'schema', 'application', 'catalog', 'interactive resource', 'repository', 'training platform', 'website', 'database', 'directory', 'encyclopedia', 'gazetteer', 'knowledge organization system', 'software library', 'source code', 'computational notebook', 'bibliographic data', 'collection', 'dataset', 'dictionary', 'manuscript', 'map', 'musical notation', 'newspaper', 'newspaper article', 'database', 'directory', 'encyclopedia', 'gazetteer', 'knowledge organization system', 'computational notebook', 'research protocol', 'workflow', 'computational notebook', 'bibliography', 'article', 'blog post', 'book', 'book part', 'book chapter', 'conference output', 'data management plan', 'journal', 'report', 'research publication', 'technical documentation', 'manuscript', 'map']


In [8]:
for concept in top_concepts:
    if concept in sub_concepts:
        print(concept, "also present in sub-concepts")

dataset also present in sub-concepts
workflow also present in sub-concepts


In [9]:
from collections import Counter

counts = Counter(sub_concepts)
duplicates = {item: count for item, count in counts.items() if count > 1}

pprint(duplicates)

{'computational notebook': 3,
 'database': 2,
 'directory': 2,
 'encyclopedia': 2,
 'gazetteer': 3,
 'knowledge organization system': 2,
 'manuscript': 2,
 'map': 2}


### Create vocabulary

In [ ]:
seen_uris = []

top_concept = ""

for _, row in df.iterrows():

    # Create URIs from the cleaned labels, lowering the case and replacing spaces with hyphens
    if pd.isna(row["uri"]):
        uri = row["label_clean"].lower().replace(" ","-").strip()
        uri = URIRef("https://vocabs.dariah.eu/resource-type/" + uri)
    else:
        uri = URIRef("https://vocabs.dariah.eu/resource-type/" + row["uri"])

    if uri in seen_uris:
        if pd.isna(row["top"]):
            voc.add((uri, SKOS.broader, top_concept))
        continue
    
    else:
        voc.add((uri, RDF.type, SKOS.Concept))
        seen_uris.append(uri)

        # prefLabel
        voc.add((uri, SKOS.prefLabel, Literal(row["label_clean"], lang='en')))

        # exactMatch
        if pd.notna(row["exact_match"]):
            voc.add((uri, SKOS.exactMatch, URIRef(row["exact_match"])))
            voc.add((uri, DC.source, URIRef(row["exact_match"])))

        # definition
        voc.add((uri, SKOS.definition, Literal(row["definition"], lang='en')))

        # comments
        if pd.notna(row["comments"]):
            voc.add((uri, SKOS.editorialNote, Literal(row["comments"], lang='en')))

        # closeMatches
        if pd.notna(row["close_match"]):
            for match in row["close_match"].split(";"):
                voc.add((uri, SKOS.closeMatch, URIRef(match.strip())))

        # declare it as topConcept or associate with broader concept
        if pd.notna(row["top"]):
            voc.add((uri, SKOS.topConceptOf, concept_scheme))
            top_concept = uri  
        else:
            voc.add((uri, SKOS.broader, top_concept))

voc.serialize(destination="vocabulary.ttl")

<Graph identifier=N7f51e4dd947a42afa1da9772a07f4a33 (<class 'rdflib.graph.Graph'>)>